In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, date_format, round

# Read raw Selic table
df = spark.table("portfolio.bcb_economic_indicators.raw_selic_daily")

# Transform and enrich data
silver_df = (
    df
    .select(
        col("data").alias("date"),
        col("valor").alias("selic_rate"),
        col("series_code"),
        col("series_name"),
        col("ingestion_timestamp")
    )
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("selic_rate", round(col("selic_rate"), 4))
    .orderBy("date")
)

# Save as Delta table
silver_df.write.mode("overwrite").saveAsTable(
    "portfolio.bcb_economic_indicators.silver_selic_daily"
)

display(silver_df)